# WDC Merged N-Quads — Full Corpus Analysis

Same assessment metrics as `wdc_analysis_by_class`, applied to `output/merged.nq` (all domains, all classes, deduplicated).

## 1. Setup

In [2]:
import re, json, requests
from collections import defaultdict, Counter
from pathlib import Path
from urllib.parse import urlparse
import pandas as pd
import statistics

SO_HTTP  = "http://schema.org/"
SO_HTTPS = "https://schema.org/"
RDF_TYPE = "http://www.w3.org/1999/02/22-rdf-syntax-ns#type"

MERGED_FILE = Path("output/merged.nq")
DATA_DIR    = Path("wdc_class_data")
DATA_DIR.mkdir(exist_ok=True)

assert MERGED_FILE.exists(), f"Run the merge cell in wdc_lookup_extractor.ipynb first"
print(f"Merged file : {MERGED_FILE}  ({MERGED_FILE.stat().st_size / 1e6:.1f} MB)")

def normalize_schema_type(uri):
    return re.sub(r"^https?://(?:www\.)?schema\.org/", "", uri)

Merged file : output/merged.nq  (982.2 MB)


## 2. Parse merged.nq → entity store

In [3]:
NQ_RE = re.compile(
    r'^'
    r'(<[^>]*>|_:\S+)'
    r'\s+'
    r'(<[^>]+>)'
    r'\s+'
    r'(<[^>]*>|_:\S+|"(?:[^"\\]|\\.)*"(?:@\w[\w-]*|(?:\^\^<[^>]+>))?)'
    r'\s+'
    r'(<[^>]+>)'
    r'\s+\.\s*$'
)

def parse_nq_line(line):
    m = NQ_RE.match(line)
    return (m.group(1), m.group(2), m.group(3), m.group(4)) if m else None

def strip_angle(s):
    return s[1:-1] if s.startswith("<") and s.endswith(">") else s

def build_entity_store(nq_path):
    def make_entity():
        return {"types": set(), "props": defaultdict(lambda: defaultdict(set)), "graphs": set()}

    store = defaultdict(make_entity)
    parse_errors = total = 0

    with open(nq_path, encoding="utf-8", errors="replace") as f:
        for line in f:
            total += 1
            parsed = parse_nq_line(line.rstrip("\n"))
            if not parsed:
                parse_errors += 1
                continue
            subj, pred, obj, graph = parsed
            graph_uri = strip_angle(graph)
            es = store[subj]
            es["graphs"].add(graph_uri)
            pred_uri = strip_angle(pred)
            if pred_uri == RDF_TYPE:
                es["types"].add(strip_angle(obj))
            else:
                es["props"][pred_uri][obj].add(graph_uri)
            if total % 500_000 == 0:
                print(f"  {total:,} lines …")

    return dict(store), parse_errors, total


print("Building entity store from merged.nq …")
store, n_errors, n_total = build_entity_store(MERGED_FILE)
print(f"  Lines parsed : {n_total:,}")
print(f"  Parse errors : {n_errors:,}  ({n_errors/n_total*100:.2f}%)")
print(f"  Entities     : {len(store):,}")

Building entity store from merged.nq …
  500,000 lines …
  1,000,000 lines …
  1,500,000 lines …
  2,000,000 lines …
  2,500,000 lines …
  3,000,000 lines …
  3,500,000 lines …
  4,000,000 lines …
  Lines parsed : 4,153,230
  Parse errors : 2  (0.00%)
  Entities     : 807,601


## 3. Load schema.org vocabulary

In [4]:
VOCAB_CACHE = DATA_DIR / "schemaorg_vocab.json"

def load_schemaorg_vocab():
    if VOCAB_CACHE.exists():
        print("Loading vocab from cache …")
        with open(VOCAB_CACHE) as f:
            raw = json.load(f)
    else:
        url = "https://schema.org/version/latest/schemaorg-current-https.jsonld"
        print(f"Fetching {url} …")
        r = requests.get(url, timeout=60)
        r.raise_for_status()
        raw = r.json()
        with open(VOCAB_CACHE, "w") as f:
            json.dump(raw, f)

    graph = raw.get("@graph", [])

    def local(uri):
        return uri.split("/")[-1].split("#")[-1].split(":")[-1]

    def ids_of(val):
        if isinstance(val, dict):  return [val.get("@id", "")]
        if isinstance(val, list):  return [v.get("@id", "") if isinstance(v, dict) else str(v) for v in val]
        return [str(val)]

    all_types    = set()
    type_props   = defaultdict(set)
    prop_range   = defaultdict(set)
    type_parents = defaultdict(set)

    for item in graph:
        item_type = str(item.get("@type", ""))
        if "rdfs:Class" in item_type or "schema:DataType" in item_type:
            name = local(item.get("@id", ""))
            all_types.add(name)
            for pid in ids_of(item.get("rdfs:subClassOf", [])):
                if pid: type_parents[name].add(local(pid))

    for item in graph:
        if "rdf:Property" in str(item.get("@type", "")):
            pname = local(item.get("@id", ""))
            for did in ids_of(item.get("schema:domainIncludes", [])):
                if did: type_props[local(did)].add(pname)
            for rid in ids_of(item.get("schema:rangeIncludes", [])):
                if rid: prop_range[pname].add(local(rid))

    memo = {}
    def get_ancestors(t):
        if t in memo: return memo[t]
        anc = set()
        for p in type_parents.get(t, set()):
            anc.add(p); anc |= get_ancestors(p)
        memo[t] = anc
        return anc

    for t in list(all_types):
        for anc in get_ancestors(t):
            type_props[t] |= type_props.get(anc, set())

    print(f"  Classes : {len(all_types):,}")
    return all_types, dict(type_props), dict(prop_range), dict(type_parents)


so_types, so_type_props, so_prop_range, so_type_parents = load_schemaorg_vocab()

Loading vocab from cache …
  Classes : 979


## 4. Namespace correctness

In [5]:
def namespace_report(entities, label):
    canonical = wrong = other = 0
    wrong_examples = Counter()
    for subj, data in entities.items():
        for t in data["types"]:
            if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS): canonical += 1
            elif re.search(r'schema\.(org|org/)', t, re.I): wrong += 1; wrong_examples[t] += 1
            else: other += 1
        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS): canonical += 1
            elif re.search(r'schema\.(org|org/)', prop, re.I): wrong += 1; wrong_examples[prop] += 1
            else: other += 1

    total_schema = canonical + wrong
    rate = canonical / total_schema * 100 if total_schema else float("nan")
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Canonical schema.org URIs : {canonical:>10,}")
    print(f"  Malformed schema.org URIs : {wrong:>10,}")
    print(f"  Other namespace URIs      : {other:>10,}  (rdf:, owl:, etc.)")
    print(f"  Namespace correctness     : {rate:.2f}%")
    if wrong_examples:
        print(f"  Malformed examples (top 5):")
        for uri, cnt in wrong_examples.most_common(5):
            print(f"    [{cnt:>5}]  {uri}")
    return rate

namespace_report(store, "All entities (merged.nq)")


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Canonical schema.org URIs :  3,860,559
  Malformed schema.org URIs :      7,020
  Other namespace URIs      :      1,405  (rdf:, owl:, etc.)
  Namespace correctness     : 99.82%
  Malformed examples (top 5):
    [ 7020]  https://www.schema.org/SiteNavigationElement


99.81849110257347

schema.org doesn't use www

## 5. Type validity

In [7]:
def type_validity_report(entities, label):
    valid = Counter()
    invalid = Counter()
    for subj, data in entities.items():
        for t in data["types"]:
            if not (t.startswith(SO_HTTP) or t.startswith(SO_HTTPS)): continue
            local_t = normalize_schema_type(t)
            if local_t in so_types: valid[local_t] += 1
            else: invalid[local_t] += 1

    total = sum(valid.values()) + sum(invalid.values())
    rate  = sum(valid.values()) / total * 100 if total else float("nan")
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid types   : {sum(valid.values()):>10,}  ({rate:.2f}%)")
    print(f"  Invalid types : {sum(invalid.values()):>10,}  ({100-rate:.2f}%)")
    print(f"\n  Valid type distribution (top 20):")
    for t, c in valid.most_common(20):
        print(f"    {t:<45} {c:>9,}")
    if invalid:
        print(f"\n  Invalid types (top 10):")
        for t, c in invalid.most_common(10):
            print(f"    {t:<45} {c:>9,}")
    return rate, valid, invalid

type_rate, valid_types, invalid_types = type_validity_report(store, "All entities (merged.nq)")


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Valid types   :    799,768  (100.00%)
  Invalid types :          0  (0.00%)

  Valid type distribution (top 20):
    Organization                                    262,842
    ImageObject                                     173,887
    Article                                          69,588
    Person                                           59,398
    WebPage                                          58,916
    NewsArticle                                      58,059
    NewsMediaOrganization                            30,890
    VideoObject                                      16,267
    WebPageElement                                   15,085
    ListItem                                          9,429
    MediaObject                                       9,123
    BlogPosting                                       6,642
    BreadcrumbL

## 6. Property validity

In [8]:
def prop_validity_report(entities, label):
    valid_props   = Counter()
    invalid_props = Counter()
    for subj, data in entities.items():
        entity_types = {
            normalize_schema_type(t)
            for t in data["types"]
            if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS)
        }
        allowed = set()
        for et in entity_types:
            allowed |= so_type_props.get(et, set())
        for prop in data["props"]:
            if not (prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)): continue
            prop_local = normalize_schema_type(prop)
            if prop_local in allowed: valid_props[prop_local] += 1
            else: invalid_props[prop_local] += 1

    total = sum(valid_props.values()) + sum(invalid_props.values())
    rate  = sum(valid_props.values()) / total * 100 if total else float("nan")
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Valid property uses   : {sum(valid_props.values()):>10,}  ({rate:.2f}%)")
    print(f"  Invalid property uses : {sum(invalid_props.values()):>10,}  ({100-rate:.2f}%)")
    print(f"\n  Most used valid properties (top 20):")
    for p, c in valid_props.most_common(20):
        print(f"    {p:<45} {c:>9,}")
    if invalid_props:
        print(f"\n  Invalid properties (top 10):")
        for p, c in invalid_props.most_common(10):
            print(f"    {p:<45} {c:>9,}")
    return rate, valid_props, invalid_props

prop_rate, valid_props, invalid_props = prop_validity_report(store, "All entities (merged.nq)")


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Valid property uses   :  3,041,488  (99.37%)
  Invalid property uses :     19,303  (0.63%)

  Most used valid properties (top 20):
    name                                            473,698
    url                                             327,233
    dateModified                                    212,587
    headline                                        163,369
    datePublished                                   157,119
    logo                                            150,484
    publisher                                       149,852
    author                                          140,993
    image                                           121,868
    height                                          104,172
    width                                           104,172
    sameAs                                          103,0

## 7. Range violations

In [9]:
DATE_PROPS = {"datePublished", "dateModified", "dateCreated", "expires", "startDate", "endDate", "uploadDate"}
URL_PROPS  = {"url", "sameAs", "mainEntityOfPage", "image", "thumbnail"}
LITERAL_DATATYPES = {"Text", "Number", "Integer", "Float", "Boolean", "Date", "DateTime", "Time", "URL", "Duration"}

ISO_DATE_RE = re.compile(
    r'^\d{4}(?:-\d{2}(?:-\d{2}(?:T\d{2}:\d{2}(?::\d{2}(?:\.\d+)?)?(?:Z|[+-]\d{2}:\d{2})?)?)?)?$'
)
LANG_RE = re.compile(r"^[a-zA-Z]{2,3}(-[a-zA-Z0-9]{2,8})*$")

def is_literal(obj): return obj.startswith('"')
def is_uri(obj):     return obj.startswith('<')
def is_blank(obj):   return obj.startswith('_:')

def strip_literal(obj):
    m = re.match(r'^"((?:[^"\\]|\\.)*)', obj)
    return m.group(1) if m else obj

def validate_literal(val, ranges, prop):
    val = val.strip()
    if "Text" in ranges: return True
    if "URL" in ranges: return val.startswith("http")
    if "Boolean" in ranges: return val in {"true", "false", "0", "1"}
    if "Integer" in ranges: return re.fullmatch(r"-?\d+", val) is not None
    if "Float" in ranges or "Number" in ranges: return re.fullmatch(r"-?\d+(\.\d+)?", val) is not None
    if "Date" in ranges or "DateTime" in ranges:
        return ISO_DATE_RE.match(val.replace(" ", "T")) is not None
    if prop == "inLanguage": return LANG_RE.match(val) is not None
    return False

def range_violation_report(entities, label):
    violations = Counter()
    ok = Counter()
    viol_examples = defaultdict(list)

    for subj, data in entities.items():
        for prop, values in data["props"].items():
            if not (prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)): continue
            prop_local    = normalize_schema_type(prop)
            ranges        = so_prop_range.get(prop_local, set())
            literal_ranges = ranges & LITERAL_DATATYPES
            entity_ranges  = ranges - LITERAL_DATATYPES
            for obj in values:
                valid = False
                if is_literal(obj):
                    if literal_ranges:
                        valid = validate_literal(strip_literal(obj), literal_ranges, prop_local)
                elif is_uri(obj) or is_blank(obj):
                    valid = bool(entity_ranges) or ("URL" in literal_ranges)
                if valid: ok[prop_local] += 1
                else:
                    violations[prop_local] += 1
                    if len(viol_examples[prop_local]) < 3:
                        viol_examples[prop_local].append(
                            f"lit={strip_literal(obj)[:60]!r}" if is_literal(obj) else f"uri={obj[:60]!r}"
                        )

    total = sum(violations.values()) + sum(ok.values())
    viol_rate = sum(violations.values()) / total * 100 if total else float("nan")
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Checked value assertions : {total:>10,}")
    print(f"  Range violations         : {sum(violations.values()):>10,}  ({viol_rate:.2f}%)")
    print(f"\n  Top violation properties (top 15):")
    for p, c in violations.most_common(15):
        print(f"    {p:<35} {c:>8,}  e.g. {viol_examples.get(p, [])}")
    return viol_rate, violations

range_rate, violations = range_violation_report(store, "All entities (merged.nq)")


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Checked value assertions :  3,281,512
  Range violations         :    350,421  (10.68%)

  Top violation properties (top 15):
    height                               104,175  e.g. ["lit='955'", "lit='1403'", "lit='956'"]
    width                                104,175  e.g. ["lit='1700'", "lit='2500'", "lit='1700'"]
    mainEntity                            69,442  e.g. ["lit='http://www.wikidata.org/entity/Q44403'", "lit='http://www.wikidata.org/entity/Q5432'", "lit='http://www.wikidata.org/entity/Q150652'"]
    duration                              15,887  e.g. ["lit='PT0H1M47S'", "lit='PT0H2M34S'", "lit='PT0H0M37S'"]
    cssSelector                           15,857  e.g. ["lit='.article__content'", "lit='.article-header'", "lit='.article-header'"]
    partner                               15,085  e.g. ["uri='_:ncfcca163d5b74056bba6

## 8. Prevalence — entities per domain

In [11]:
def get_pld(url):
    try:
        host  = urlparse(url).netloc or url
        parts = host.split(".")
        return ".".join(parts[-2:]) if len(parts) >= 2 else host
    except Exception:
        return url

def prevalence_report(entities, label, top_n=30):
    domain_counts = Counter()
    for subj, data in entities.items():
        seen = set()
        for graph in data.get("graphs", set()) or {subj}:
            d = get_pld(graph)
            if d not in seen:
                seen.add(d)
                domain_counts[d] += 1

    total = sum(domain_counts.values())
    print(f"\n{'─'*60}")
    print(f"  {label}")
    print(f"{'─'*60}")
    print(f"  Total entity-domain pairs : {total:,}")
    print(f"  Unique domains            : {len(domain_counts):,}")
    print(f"\n  Top {top_n} domains by entity count:")
    df = pd.DataFrame(domain_counts.most_common(top_n), columns=["domain", "entities"])
    df["% of total"] = (df["entities"] / total * 100).round(2)
    print(df.to_string(index=False))
    return domain_counts

domain_counts = prevalence_report(store, "All entities (merged.nq)")


────────────────────────────────────────────────────────────
  All entities (merged.nq)
────────────────────────────────────────────────────────────
  Total entity-domain pairs : 807,601
  Unique domains            : 10

  Top 30 domains by entity count:
         domain  entities  % of total
  wikipedia.org    281360       34.84
        cnn.com    281225       34.82
         dw.com    208339       25.80
  factcheck.org     17546        2.17
   fullfact.org      9442        1.17
 politifact.com      4660        0.58
  aljazeera.com      2974        0.37
   france24.com      1647        0.20
   thetimes.com       390        0.05
theguardian.com        18        0.00


In [24]:
# ── Average number of distinct schema.org types per page, per domain ──

page_type_counts = []

for graph, subjects in page_to_subjects.items():
    types_on_page = set()
    for subj in subjects:
        for t in store[subj]["types"]:
            if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS):
                types_on_page.add(normalize_schema_type(t))
    host, reg = _parse_host(graph)
    page_type_counts.append({
        "uri":    graph,
        "domain": reg,
        "n_types": len(types_on_page),
        "types":  types_on_page,
    })

df_pt = pd.DataFrame(page_type_counts)

print(f"{'='*65}")
print("Avg distinct schema.org types per page — by domain")
print(f"{'='*65}")
print(f"  {'domain':<35} {'pages':>7}  {'avg_types':>10}  {'median':>7}  {'max':>5}")
print(f"  {'─'*63}")

summary = (
    df_pt.groupby("domain")
    .agg(
        pages     = ("uri",     "count"),
        avg_types = ("n_types", "mean"),
        median    = ("n_types", "median"),
        max_types = ("n_types", "max"),
    )
    .sort_values("pages", ascending=False)
)

for domain, row in summary.iterrows():
    print(f"  {domain:<35} {row['pages']:>7,}  {row['avg_types']:>10.2f}  "
          f"{row['median']:>7.1f}  {row['max_types']:>5.0f}")

# ── Overall ───────────────────────────────────────────────────────
print(f"\n  {'OVERALL':<35} {len(df_pt):>7,}  {df_pt['n_types'].mean():>10.2f}  "
      f"{df_pt['n_types'].median():>7.1f}  {df_pt['n_types'].max():>5}")

# ── Distribution across all pages ────────────────────────────────
print(f"\n{'='*60}")
print("Distribution of type count per page (all domains)")
print(f"{'='*60}")
dist = Counter(df_pt["n_types"])
bar_max = max(dist.values())
for k in sorted(dist):
    bar = "█" * (dist[k] * 40 // bar_max)
    print(f"  {k:>3} types: {bar:<40} {dist[k]:>8,}  ({dist[k]/len(df_pt)*100:>5.1f}%)")


Avg distinct schema.org types per page — by domain
  domain                                pages   avg_types   median    max
  ───────────────────────────────────────────────────────────────
  wikipedia.org                       66,739.0        3.00      3.0      6
  dw.com                              41,245.0        4.51      5.0      5
  cnn.com                             18,825.0        7.81      8.0     15
  factcheck.org                       1,602.0        7.39      7.0     11
  fullfact.org                        1,158.0        5.66      8.0      9
  politifact.com                        725.0        5.00      5.0      5
  aljazeera.com                         334.0        5.21      5.0     11
  france24.com                          237.0        6.91      7.0      7
  thetimes.com                          118.0        2.66      2.0      5
  theguardian.com                         2.0        2.00      2.0      2

  OVERALL                             130,985        4.27      3.

## 9. Property coverage — per top schema.org type

In [12]:
def coverage_report(entities, target_class):
    defined_props = so_type_props.get(target_class, set())
    if not defined_props:
        print(f"  No properties defined for {target_class}")
        return

    # Filter to entities of this type
    typed_entities = {
        s: d for s, d in entities.items()
        if any(normalize_schema_type(t) == target_class for t in d["types"])
    }
    if not typed_entities:
        print(f"  No entities of type {target_class}")
        return

    prop_freq = Counter()
    for data in typed_entities.values():
        for prop in data["props"]:
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS):
                p = normalize_schema_type(prop)
                if p in defined_props:
                    prop_freq[p] += 1

    used      = set(prop_freq.keys())
    coverage  = len(used) / len(defined_props) * 100
    n         = len(typed_entities)

    print(f"\n{'─'*60}")
    print(f"  {target_class} — Property Coverage  ({n:,} entities)")
    print(f"{'─'*60}")
    print(f"  Properties defined : {len(defined_props):>5}")
    print(f"  Properties used    : {len(used):>5}  ({coverage:.1f}%)")
    print(f"  Properties unused  : {len(defined_props)-len(used):>5}  ({100-coverage:.1f}%)")

    rows = [(p, prop_freq[p], prop_freq[p]/n*100) for p in defined_props]
    rows.sort(key=lambda x: -x[1])
    df = pd.DataFrame(rows, columns=["property", "entities_using", "density_%"])
    df["density_%"] = df["density_%"].round(2)
    print(f"\n  Properties used (sorted by frequency):")
    print(df[df["entities_using"] > 0].to_string(index=False))
    unused = sorted(defined_props - used)
    print(f"\n  Never-used: {unused}")


# Run for the top types by declaration count
TOP_TYPES = [t for t, _ in valid_types.most_common(10)]
print(f"Running coverage for top types: {TOP_TYPES}\n")
for cls in TOP_TYPES:
    coverage_report(store, cls)

Running coverage for top types: ['Organization', 'ImageObject', 'Article', 'Person', 'WebPage', 'NewsArticle', 'NewsMediaOrganization', 'VideoObject', 'WebPageElement', 'ListItem']


────────────────────────────────────────────────────────────
  Organization — Property Coverage  (262,842 entities)
────────────────────────────────────────────────────────────
  Properties defined :    89
  Properties used    :     5  (5.6%)
  Properties unused  :    84  (94.4%)

  Properties used (sorted by frequency):
property  entities_using  density_%
    name          262840     100.00
    logo          119594      45.50
     url           22092       8.41
  sameAs            1247       0.47
   image            1009       0.38

  Never-used: ['acceptedPaymentMethod', 'actionableFeedbackPolicy', 'additionalType', 'address', 'agentInteractionStatistic', 'aggregateRating', 'alternateName', 'alumni', 'areaServed', 'award', 'awards', 'brand', 'companyRegistration', 'contactPoint', 'contactPoints', 'correc

## 10. Average properties per entity (AvP)

In [13]:
def avp_report(entities, label, target_class=None):
    defined_props = so_type_props.get(target_class, set()) if target_class else set()
    n_defined = len(defined_props) or "N/A"

    prop_counts = []
    for data in entities.values():
        so_props = {
            normalize_schema_type(prop)
            for prop in data["props"]
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        }
        prop_counts.append(len(so_props))

    if not prop_counts:
        print(f"  No entities for {label}"); return

    avp = statistics.mean(prop_counts)
    med = statistics.median(prop_counts)
    std = statistics.stdev(prop_counts) if len(prop_counts) > 1 else 0
    mn, mx = min(prop_counts), max(prop_counts)

    print(f"\n{'─'*60}")
    print(f"  {label} — Average Properties per Entity (AvP)")
    print(f"{'─'*60}")
    print(f"  Entities        : {len(prop_counts):>10,}")
    print(f"  AvP (mean)      : {avp:>10.2f}")
    print(f"  Median          : {med:>10.1f}")
    print(f"  Std dev         : {std:>10.2f}")
    print(f"  Min / Max       : {mn:>4}  /  {mx:>4}")

    buckets = Counter(min(c, 20) for c in prop_counts)
    print(f"\n  Distribution:")
    bar_max = max(buckets.values())
    for k in range(0, min(mx + 2, 22)):
        cnt = buckets.get(k, 0)
        bar = '█' * (cnt * 40 // bar_max)
        label_k = f"{k:>2}" if k < 20 else "20+"
        print(f"    {label_k}: {bar:<40} {cnt:>9,}  ({cnt/len(prop_counts)*100:>5.1f}%)")


# All entities
avp_report(store, "All entities (merged.nq)")

# AvP breakdown by schema.org type
print(f"\n\n{'═'*60}")
print("  AvP breakdown by schema.org type")
print(f"{'═'*60}")
print(f"  {'type':<35} {'entities':>9}  {'avail':>6}  {'AvP':>6}  {'median':>7}")
print(f"  {'─'*65}")
for cls, _ in valid_types.most_common(20):
    typed = [
        len({
            normalize_schema_type(p)
            for p in d["props"]
            if p.startswith(SO_HTTP) or p.startswith(SO_HTTPS)
        })
        for d in store.values()
        if any(normalize_schema_type(t) == cls for t in d["types"])
    ]
    if not typed: continue
    n_defined = len(so_type_props.get(cls, set()))
    print(f"  {cls:<35} {len(typed):>9,}  {n_defined:>6}  {statistics.mean(typed):>6.2f}  {statistics.median(typed):>7.1f}")



────────────────────────────────────────────────────────────
  All entities (merged.nq) — Average Properties per Entity (AvP)
────────────────────────────────────────────────────────────
  Entities        :    807,601
  AvP (mean)      :       3.79
  Median          :        2.0
  Std dev         :       3.76
  Min / Max       :    0  /    19

  Distribution:
     0: ████████                                    48,287  (  6.0%)
     1: ████████████████████████████████████████   223,503  ( 27.7%)
     2: ████████████████████████████████████       205,475  ( 25.4%)
     3: ████████████                                70,859  (  8.8%)
     4:                                              4,706  (  0.6%)
     5: ██████                                      34,565  (  4.3%)
     6: ████                                        23,128  (  2.9%)
     7: ██████████                                  57,152  (  7.1%)
     8: ██                                          13,906  (  1.7%)
     9: ███████ 

## 11. Characteristic property sets

In [14]:
def characteristic_sets_report(entities, label, top_n=20):
    set_counter = Counter()
    for data in entities.values():
        so_props = frozenset(
            normalize_schema_type(prop)
            for prop in data["props"]
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        )
        set_counter[so_props] += 1

    total = len(entities) or 1
    print(f"\n{'─'*60}")
    print(f"  {label} — Characteristic Sets (top {top_n})")
    print(f"{'─'*60}")
    print(f"  Unique property sets : {len(set_counter):,}")
    print()
    for i, (prop_set, count) in enumerate(set_counter.most_common(top_n), 1):
        pct = count / total * 100
        props_str = ", ".join(sorted(prop_set)) if prop_set else "(no SO properties)"
        print(f"  #{i:>2}  [{count:>8,}  {pct:>5.1f}%]  {{{props_str}}}")
    return set_counter

# Per class — top 20 types by declaration count
for cls, _ in valid_types.most_common(20):
    typed_entities = {
        s: d for s, d in store.items()
        if any(normalize_schema_type(t) == cls for t in d["types"])
    }
    if not typed_entities:
        continue
    characteristic_sets_report(typed_entities, cls)



────────────────────────────────────────────────────────────
  Organization — Characteristic Sets (top 20)
────────────────────────────────────────────────────────────
  Unique property sets : 11

  # 1  [ 121,708   46.3%]  {name}
  # 2  [ 118,679   45.2%]  {logo, name}
  # 3  [  20,185    7.7%]  {name, url}
  # 4  [     990    0.4%]  {image, name, url}
  # 5  [     900    0.3%]  {logo, name, sameAs, url}
  # 6  [     346    0.1%]  {name, sameAs}
  # 7  [      17    0.0%]  {image, name}
  # 8  [      13    0.0%]  {logo, name, url}
  # 9  [       2    0.0%]  {url}
  #10  [       1    0.0%]  {image, logo, name, url}
  #11  [       1    0.0%]  {image, logo, name, sameAs, url}

────────────────────────────────────────────────────────────
  ImageObject — Characteristic Sets (top 20)
────────────────────────────────────────────────────────────
  Unique property sets : 26

  # 1  [  69,507   40.0%]  {url}
  # 2  [  52,114   30.0%]  {height, url, width}
  # 3  [  19,188   11.0%]  {contentUrl,

## 12. Nesting depth

In [15]:
LITERAL_DATATYPES = {"Text", "Number", "Integer", "Float", "Boolean", "Date", "DateTime", "Time", "URL", "Duration"}

def nesting_depth_report(entities, full_store, label):
    def _max_depth(key, visited):
        if key in visited or key not in full_store: return 0
        visited.add(key)
        best = 0
        for obj_dict in full_store[key]["props"].values():
            for obj_val in obj_dict:
                if obj_val in full_store:
                    best = max(best, _max_depth(obj_val, visited) + 1)
        return best

    def _direct_refs(key):
        bn = uri = 0
        if key not in full_store: return bn, uri
        for obj_dict in full_store[key]["props"].values():
            for obj_val in obj_dict:
                if obj_val in full_store:
                    if obj_val.startswith("_:"): bn += 1
                    else: uri += 1
        return bn, uri

    print(f"Computing depths for {len(entities):,} entities …")
    depths = [_max_depth(s, set()) for s in entities]

    n      = len(depths)
    avg    = statistics.mean(depths)
    med    = statistics.median(depths)
    mx     = max(depths)
    std    = statistics.stdev(depths) if n > 1 else 0.0
    srt    = sorted(depths)
    p25    = srt[int(n * 0.25)]
    p75    = srt[int(n * 0.75)]
    p90    = srt[int(n * 0.90)]
    p95    = srt[min(int(n * 0.95), n - 1)]
    depth_counts = Counter(depths)

    print(f"\n{'═'*62}")
    print(f"  {label} — Nesting Depth Analysis")
    print(f"{'═'*62}")
    print(f"  Entities analysed : {n:>10,}")
    print(f"  Mean depth        : {avg:>10.2f}  (σ = {std:.2f})")
    print(f"  Median depth      : {med:>10.1f}")
    print(f"  Max depth         : {mx:>10}")
    print(f"\n  Percentiles:  p25={p25}  p75={p75}  p90={p90}  p95={p95}")

    print(f"\n  Depth distribution:")
    bar_max = max(depth_counts.values(), default=1)
    for d in range(0, mx + 1):
        cnt = depth_counts.get(d, 0)
        bar = '█' * (cnt * 40 // bar_max)
        print(f"    depth {d:>2}: {bar:<40} {cnt:>9,}  ({cnt/n*100:5.1f}%)")

    flat_n   = depth_counts.get(0, 0)
    nested_n = n - flat_n
    deep_n   = sum(v for d, v in depth_counts.items() if d >= 3)
    print(f"\n  Flatness:")
    print(f"    Flat   (depth = 0) : {flat_n:>10,}  ({flat_n/n*100:.1f}%)")
    print(f"    Nested (depth ≥ 1) : {nested_n:>10,}  ({nested_n/n*100:.1f}%)")
    print(f"    Deep   (depth ≥ 3) : {deep_n:>10,}  ({deep_n/n*100:.1f}%)")

    total_bn = total_uri = 0
    for subj in entities:
        bn, uri = _direct_refs(subj)
        total_bn += bn; total_uri += uri
    total_refs = total_bn + total_uri
    print(f"\n  Cross-reference density:")
    print(f"    Total cross-refs    : {total_refs:>10,}")
    print(f"    Avg refs per entity : {total_refs/n:>10.2f}")
    if total_refs:
        print(f"    Blank-node refs     : {total_bn:>10,}  ({total_bn/total_refs*100:.1f}%)")
        print(f"    URI refs            : {total_uri:>10,}  ({total_uri/total_refs*100:.1f}%)")

    ref_type_counts = Counter()
    for subj in entities:
        if subj not in full_store: continue
        for obj_dict in full_store[subj]["props"].values():
            for obj_val in obj_dict:
                if obj_val in full_store:
                    for t in full_store[obj_val].get("types", set()):
                        ref_type_counts[normalize_schema_type(t)] += 1
                    if not full_store[obj_val].get("types"):
                        ref_type_counts["(untyped)"] += 1
    if ref_type_counts:
        print(f"\n  Types of directly referenced entities (top 15):")
        for t, c in ref_type_counts.most_common(15):
            print(f"    {t:<35} {c:>9,}")

    prop_totals = defaultdict(lambda: [0, 0])
    for subj, data in entities.items():
        for pred, obj_dict in data["props"].items():
            if not (pred.startswith(SO_HTTP) or pred.startswith(SO_HTTPS)): continue
            pname = normalize_schema_type(pred)
            for obj_val in obj_dict:
                if obj_val in full_store: prop_totals[pname][1] += 1
                else: prop_totals[pname][0] += 1

    rows = [(p, l, s) for p, (l, s) in prop_totals.items() if l + s >= 50]
    print(f"\n  Properties most often linking to structured entities:")
    print(f"  {'property':<25} {'literal':>8} {'struct':>8} {'%struct':>8}")
    for pname, lit, struct in sorted(rows, key=lambda r: r[2]/(r[1]+r[2]), reverse=True)[:15]:
        total = lit + struct
        print(f"  {pname:<25} {lit:>8,} {struct:>8,} {struct/total*100:>7.1f}%")

    print(f"\n  Literal-only enrichment candidates")
    print(f"  {'property':<25} {'uses':>8}  entity range types")
    cands = [(p, l, s) for p, l, s in rows if s == 0 and (so_prop_range.get(p, set()) - LITERAL_DATATYPES)]
    cands.sort(key=lambda r: r[1], reverse=True)
    for pname, lit, _ in cands[:15]:
        er = ", ".join(sorted((so_prop_range.get(pname, set()) - LITERAL_DATATYPES))[:4])
        print(f"  {pname:<25} {lit:>8,}  {er}")

    return depths

depths = nesting_depth_report(store, store, "All entities (merged.nq)")

Computing depths for 807,601 entities …

══════════════════════════════════════════════════════════════
  All entities (merged.nq) — Nesting Depth Analysis
══════════════════════════════════════════════════════════════
  Entities analysed :    807,601
  Mean depth        :       0.67  (σ = 0.87)
  Median depth      :        0.0
  Max depth         :          7

  Percentiles:  p25=0  p75=1  p90=2  p95=2

  Depth distribution:
    depth  0: ████████████████████████████████████████   450,833  ( 55.8%)
    depth  1: █████████████████                          195,528  ( 24.2%)
    depth  2: ████████████                               142,481  ( 17.6%)
    depth  3: █                                           14,550  (  1.8%)
    depth  4:                                              2,494  (  0.3%)
    depth  5:                                              1,351  (  0.2%)
    depth  6:                                                363  (  0.0%)
    depth  7:                                

## 13. Deeply nested entities — examples

In [16]:
def fmt_types(s, key):
    types = s.get(key, {}).get("types", set())
    return ", ".join(normalize_schema_type(t) for t in types) or "untyped"

top5 = sorted(zip(store, depths), key=lambda x: x[1], reverse=True)[:5]
for subj, depth in top5:
    print(f"\n{'═'*70}")
    print(f"  subject : {subj}")
    print(f"  types   : {fmt_types(store, subj)}")
    print(f"  depth   : {depth}")
    print(f"  graphs  : {store[subj]['graphs']}")
    print(f"  props:")
    for pred, obj_dict in store[subj]["props"].items():
        pname = normalize_schema_type(pred)
        for obj_val in obj_dict:
            if obj_val in store:
                print(f"    {pname:<25} → [{fmt_types(store, obj_val)}]  {obj_val[:60]}")
                for pred2, obj_dict2 in store[obj_val]["props"].items():
                    pname2 = normalize_schema_type(pred2)
                    for ov2 in obj_dict2:
                        if ov2 in store:
                            print(f"      {pname2:<23} → [{fmt_types(store, ov2)}]  {ov2[:58]}")
                        else:
                            print(f"      {pname2:<23} = {ov2[:78]}")
            else:
                print(f"    {pname:<25} = {obj_val[:80]}")


══════════════════════════════════════════════════════════════════════
  subject : _:nc8acf2c087b144049729c86c8c6155e8xb0
  types   : NewsArticle
  depth   : 7
  graphs  : {'https://cnnespanol.cnn.com/2020/10/15/signos-piel-pueden-indicar-que-una-persona-tiene-coronavirus-covid-19/'}
  props:
    articleBody               = "(CNN Español) – El Dr. Elmer Huerta explica los diferentes tipos de lesiones en
    articleSection            = "salud"
    author                    → [Person]  _:nc8acf2c087b144049729c86c8c6155e8xb1
      name                    = "Dr. Elmer Huerta"
      url                     → [WebPage]  <https://cnnespanol.cnn.com/2020/10/15/signos-piel-pueden-
    dateModified              = "2020-10-15T14:00:55Z"^^<http://schema.org/Date>
    datePublished             = "2020-10-15T14:00:55Z"^^<http://schema.org/Date>
    description               = "El Dr. Elmer Huerta explica los diferentes tipos de lesiones en la piel causada
    hasPart                   → [WebPageEle

In [17]:
def _parse_host(uri):
    try:
        host = urlparse(uri).netloc.lower()
        parts = host.split(".")
        if len(parts) >= 3 and parts[-2] in ("co", "com", "org", "net", "edu", "gov", "ac"):
            reg = ".".join(parts[-3:])
        else:
            reg = ".".join(parts[-2:]) if len(parts) >= 2 else host
        return host, reg
    except Exception:
        return None, None


In [ ]:
# ── Schema.org property fill rate per entity, page, and domain ───

# For each entity: how many of its type-defined properties are used?
fill_records = []

for subj, data in store.items():
    # Collect all schema.org types for this entity
    entity_types = {
        normalize_schema_type(t)
        for t in data["types"]
        if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS)
    }
    if not entity_types:
        continue

    # Union of all properties defined for its types (via vocab)
    available = set()
    for t in entity_types:
        available |= so_type_props.get(t, set())
    if not available:
        continue

    # Properties actually used (schema.org only)
    used = {
        normalize_schema_type(prop)
        for prop in data["props"]
        if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
    } & available   # only count props that are defined for this type

    fill_rate = len(used) / len(available)

    for graph in data["graphs"]:
        host, reg = _parse_host(graph)
        fill_records.append({
            "subj":              subj,
            "uri":               graph,
            "host":              host,
            "registered_domain": reg,
            "types":             ", ".join(sorted(entity_types)),
            "primary_type":      next(iter(sorted(entity_types))),   # deterministic
            "available":         len(available),
            "used":              len(used),
            "fill_rate":         fill_rate,
        })

df_fill = pd.DataFrame(fill_records)
print(f"Entity-page pairs: {len(df_fill):,}")
print(f"Mean fill rate overall: {df_fill['fill_rate'].mean()*100:.1f}%\n")

# ── Fill rate per domain ──────────────────────────────────────────
print(f"\n{'='*65}")
print("Fill rate per registered domain (avg across all entities on pages from that domain)")
print(f"{'='*65}")
print(f"  {'domain':<35} {'entities':>9}  {'avail':>6}  {'mean_used':>10}  {'fill%':>6}")
print(f"  {'─'*70}")

domain_fill = (
    df_fill.groupby("registered_domain")
    .agg(
        entities  = ("subj",      "nunique"),
        avg_avail = ("available", "mean"),
        avg_used  = ("used",      "mean"),
        fill_rate = ("fill_rate", "mean"),
    )
    .sort_values("entities", ascending=False)
)
for domain, row in domain_fill.iterrows():
    print(
        f"  {domain:<35} {row['entities']:>9,}  "
        f"{row['avg_avail']:>6.0f}  {row['avg_used']:>10.2f}  "
        f"{row['fill_rate']*100:>5.1f}%"
    )

# ── Fill rate distribution ────────────────────────────────────────
print(f"\n{'='*60}")
print("Fill rate distribution (all entity-page pairs)")
print(f"{'='*60}")
buckets = pd.cut(
    df_fill["fill_rate"],
    bins=[0, 0.05, 0.1, 0.2, 0.3, 0.5, 1.001],
    include_lowest=True,
    labels=["0–5%", "5–10%", "10–20%", "20–30%", "30–50%", "50–100%"],
)
dist = buckets.value_counts().sort_index()
n = dist.sum()
for label, count in dist.items():
    bar = "█" * (count * 40 // dist.max())
    print(f"  {label:<10} {bar:<40} {count:>9,}  ({count/n*100:>5.1f}%)")

# ── Best and worst annotated entities (examples) ─────────────────
print(f"\n{'='*60}")
print("Most completely annotated entities (top 10 by fill rate, min 5 used props)")
print(f"{'='*60}")
top = df_fill[df_fill["used"] >= 5].nlargest(10, "fill_rate")
for _, row in top.iterrows():
    uri = row["uri"][:70] + "…" if len(row["uri"]) > 70 else row["uri"]
    print(f"  {row['fill_rate']*100:>5.1f}%  {row['used']:>3}/{row['available']:<3}  [{row['primary_type']}]  {uri}")


Entity-page pairs: 809,635
Mean fill rate overall: 3.3%


Fill rate per registered domain (avg across all entities on pages from that domain)
  domain                               entities   avail   mean_used   fill%
  ──────────────────────────────────────────────────────────────────────
  wikipedia.org                       281,293.0     116        3.41    2.8%
  cnn.com                             281,214.0     113        5.02    4.3%
  dw.com                              208,338.0     121        2.74    2.3%
  factcheck.org                        10,519.0      80        3.94    7.0%
  fullfact.org                          9,442.0      97        2.86    3.2%
  politifact.com                        3,935.0      91        4.10    8.0%
  aljazeera.com                         2,971.0     117        2.97    2.9%
  france24.com                          1,647.0      51        2.91    7.2%
  thetimes.com                            389.0      86        3.70    4.9%
  theguardian.com        

In [ ]:
# ── Co-occurring entity types on pages with both Article & NewsArticle ──────

# Step 1: build graph → subjects index
page_to_subjects = defaultdict(set)
for subj, data in store.items():
    for graph in data["graphs"]:
        page_to_subjects[graph].add(subj)

def entity_types(subj):
    return {
        normalize_schema_type(t)
        for t in store[subj]["types"]
        if t.startswith(SO_HTTP) or t.startswith(SO_HTTPS)
    }

# Step 2: pages with Article OR NewsArticle (or both on same entity)
ARTICLE_TYPES = {"Article", "NewsArticle"}

article_pages      = set()
news_article_pages = set()

for graph, subjects in page_to_subjects.items():
    for subj in subjects:
        types = entity_types(subj)
        if "Article"     in types: article_pages.add(graph)
        if "NewsArticle" in types: news_article_pages.add(graph)

target_pages = article_pages | news_article_pages   # ← OR instead of AND

print(f"Pages with Article OR NewsArticle: {len(target_pages):,}")
print(f"  Article pages     : {len(article_pages):,}")
print(f"  NewsArticle pages : {len(news_article_pages):,}")
print(f"  Overlap (both)    : {len(article_pages & news_article_pages):,}")


# Step 3: collect every co-occurring entity type on those pages
co_type_counter  = Counter()                       # type → page-count
co_type_entities = defaultdict(dict)               # type → {subj: data}  (deduped by subj)

for graph in target_pages:
    seen_types_this_page = set()
    for subj in page_to_subjects[graph]:
        types = entity_types(subj)
        for t in types:
            if t not in seen_types_this_page:
                co_type_counter[t] += 1
                seen_types_this_page.add(t)
            co_type_entities[t][subj] = store[subj]   # keep entity data for char-sets

# Step 4: report co-occurring type frequencies
ARTICLE_TYPES = {"Article", "NewsArticle"}
print(f"\n{'='*65}")
print("Co-occurring entity types on Article+NewsArticle pages")
print(f"{'='*65}")
print(f"  {'type':<35} {'pages':>8}  {'% of target pages':>18}")
print(f"  {'─'*63}")
for t, page_cnt in co_type_counter.most_common(25):
    marker = " ◄ article type" if t in ARTICLE_TYPES else ""
    print(f"  {t:<35} {page_cnt:>8,}  {page_cnt/len(target_pages)*100:>17.1f}%{marker}")

# Step 5: characteristic property sets for top co-occurring non-article types
TOP_CO_TYPES = [t for t, _ in co_type_counter.most_common(20) if t not in ARTICLE_TYPES]

print(f"\n\n{'='*65}")
print("Characteristic property sets — co-occurring types")
print("(showing how richly each companion type is annotated)")
print(f"{'='*65}")

for cls in TOP_CO_TYPES[:10]:
    entities_for_cls = co_type_entities[cls]
    total = len(entities_for_cls)

    set_counter = Counter()
    for data in entities_for_cls.values():
        prop_set = frozenset(
            normalize_schema_type(prop)
            for prop in data["props"]
            if prop.startswith(SO_HTTP) or prop.startswith(SO_HTTPS)
        )
        set_counter[prop_set] += 1

    avp = sum(len(ps) * cnt for ps, cnt in set_counter.items()) / total if total else 0
    empty_pct = set_counter.get(frozenset(), 0) / total * 100

    n_avail = len(so_type_props.get(cls, set()))
    print(f"\n{'─'*65}")
    print(f"  {cls}  ({total:,} entities | avail={n_avail} props | AvP={avp:.1f} | no-prop={empty_pct:.1f}%)")
    print(f"{'─'*65}")
    for i, (prop_set, count) in enumerate(set_counter.most_common(10), 1):
        pct = count / total * 100
        props_str = ", ".join(sorted(prop_set)) if prop_set else "(no schema.org properties)"
        print(f"  #{i:>2}  [{count:>7,}  {pct:>5.1f}%]  {{{props_str}}}")



Pages with Article OR NewsArticle: 124,607
  Article pages     : 66,856
  NewsArticle pages : 57,751
  Overlap (both)    : 0

Co-occurring entity types on Article+NewsArticle pages
  type                                   pages   % of target pages
  ───────────────────────────────────────────────────────────────
  Organization                         124,523               99.9%
  ImageObject                          124,466               99.9%
  Article                               66,856               53.7% ◄ article type
  NewsArticle                           57,751               46.3% ◄ article type
  WebPage                               56,791               45.6%
  Person                                37,446               30.1%
  NewsMediaOrganization                 15,603               12.5%
  WebPageElement                        15,071               12.1%
  VideoObject                           10,098                8.1%
  MediaObject                            3,045       